# VAE Standalone per Compressione Genomica - Dataset AdaptMap

Questo notebook implementa un Variational Autoencoder (VAE) unsupervised per:
1. Comprimere dati genomici da ~50K SNP a 128-256 dimensioni latenti
2. Creare embeddings riutilizzabili per downstream tasks (classificazione, clustering, GWAS)
3. Permettere transfer learning su nuovi dataset di capre

**Dataset**: goat SNP50 BeadChip (53,347 SNP)

**References**: 
- AdaptMap consortium (2018): https://doi.org/10.1186/s12711-018-0422-x
- Beta-VAE: Higgins et al. (2017)

## 1. Installazione Dipendenze

Esegui questa cella solo alla prima esecuzione:

In [10]:
# Decommenta per installare le dipendenze
# !pip install torch torchvision pytorch-lightning pandas-plink scikit-learn matplotlib umap-learn scipy seaborn

## 2. Import Librerie

In [11]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# PyTorch Lightning
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

# Data processing
from pandas_plink import read_plink1_bin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from scipy.stats import chi2 as chi2_dist

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Seed per riproducibilità
pl.seed_everything(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("✅ Librerie importate con successo!")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Seed set to 42


✅ Librerie importate con successo!
PyTorch version: 2.9.0+cu130
GPU disponibile: True
GPU: NVIDIA GeForce RTX 5070 Ti


## 3. Configurazione Parametri

In [12]:
class Config:
    """Configurazione centrale per tutti i parametri"""
    
    # --- PATHS ---
    DATA_PATH = "./data/ADAPTmap_genotypeTOP_20160222_full.bed"
    OUTPUT_DIR = "./output_vae_standalone"
    CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
    EMBEDDINGS_DIR = os.path.join(OUTPUT_DIR, "embeddings")
    
    
    MIND_THRESHOLD = 0.05       # Max missing rate per sample
    MAF_THRESHOLD = 0.01        # 1%
    GENO_THRESHOLD = 0.05       # 5%
    HWE_PVALUE = 0
    LD_PRUNING = True
    LD_THRESHOLD = 0.5          # r² > 0.5 (CAMBIATO)
    LD_WINDOW_KB = 50
    
    # --- MODEL ARCHITECTURE ---
    LATENT_DIM = 128            # Dimensione spazio latente
    HIDDEN_DIMS = [2048, 512]   # Layers encoder/decoder
    DROPOUT_RATE = 0.1
    
    # --- TRAINING ---
    BATCH_SIZE = 64
    MAX_EPOCHS = 150
    LEARNING_RATE = 0.0001
    BETA = 0.0001               # KL weight (Beta-VAE: prova 0.001-0.01)
    
    # --- HARDWARE ---
    NUM_WORKERS = 0             # Dataloader workers (0 per Windows)
    ACCELERATOR = "auto"        # "gpu", "cpu", o "auto"
    
    def __init__(self):
        # Crea directories
        os.makedirs(self.OUTPUT_DIR, exist_ok=True)
        os.makedirs(self.CHECKPOINT_DIR, exist_ok=True)
        os.makedirs(self.EMBEDDINGS_DIR, exist_ok=True)

config = Config()
print("\n✅ Configurazione caricata:")
print(f"  Output directory: {config.OUTPUT_DIR}")
print(f"  Latent dimension: {config.LATENT_DIM}")
print(f"  Beta-VAE weight: {config.BETA}")


✅ Configurazione caricata:
  Output directory: ./output_vae_standalone
  Latent dimension: 128
  Beta-VAE weight: 0.0001


## 4. Data Module con Quality Control Completo

In [13]:
class AdaptMapDataModule(pl.LightningDataModule):
    """
    DataModule per AdaptMap con Quality Control completo secondo letteratura
    
    QC Pipeline:
    1. Sample-level filtering (MIND)
    2. SNP-level filtering (GENO, MAF, HWE)
    3. Imputation
    4. LD pruning (opzionale)
    5. Standardization
    """
    
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.imputer = SimpleImputer(strategy='mean')
        self.scaler = StandardScaler()
        
    def setup(self, stage=None):
        if hasattr(self, 'train_dataset'):
            return
        
        print("\n" + "="*80)
        print("📊 CARICAMENTO E QUALITY CONTROL DATASET ADAPTMAP")
        print("="*80)
        
        # 1. LOAD RAW DATA
        print(f"\n[1/7] Caricamento file PLINK: {self.config.DATA_PATH}")
        
        if not os.path.exists(self.config.DATA_PATH):
            raise FileNotFoundError(f"File non trovato: {self.config.DATA_PATH}")
        
        G = read_plink1_bin(self.config.DATA_PATH, verbose=False)
        fam = G.sample.to_dataframe()
        bim = G.variant.to_dataframe()
        
        X = G.values
        if X.shape[1] == len(fam):
            print("   Trasposizione matrice: SNP x Sample → Sample x SNP")
            X = X.T
        
        print(f"   Shape iniziale: {X.shape} ({X.shape[0]} samples, {X.shape[1]} SNPs)")
        print(f"   Missing rate globale: {np.isnan(X).mean():.2%}")
        
        # 2. SAMPLE QC (MIND)
        print(f"\n[2/7] Sample Quality Control (MIND < {self.config.MIND_THRESHOLD})")
        
        sample_missing_rate = np.isnan(X).mean(axis=1)
        sample_pass = sample_missing_rate < self.config.MIND_THRESHOLD
        
        n_removed_samples = (~sample_pass).sum()
        print(f"   Rimossi {n_removed_samples} samples con call rate < {1-self.config.MIND_THRESHOLD:.2%}")
        
        X = X[sample_pass]
        fam = fam[sample_pass].reset_index(drop=True)
        print(f"   ✓ Samples rimanenti: {X.shape[0]}")
        
        # 3. SNP QC (GENO)
        print(f"\n[3/7] SNP Quality Control - Missingness (GENO < {self.config.GENO_THRESHOLD})")
        
        snp_missing_rate = np.isnan(X).mean(axis=0)
        snp_pass_geno = snp_missing_rate < self.config.GENO_THRESHOLD
        
        n_removed_geno = (~snp_pass_geno).sum()
        print(f"   Rimossi {n_removed_geno} SNPs con call rate < {1-self.config.GENO_THRESHOLD:.2%}")
        
        X = X[:, snp_pass_geno]
        bim = bim[snp_pass_geno].reset_index(drop=True)
        
        # 4. IMPUTATION
        print(f"\n[4/7] Imputation valori mancanti (strategia: mean)")
        X = self.imputer.fit_transform(X)
        print(f"   ✓ Missing rate post-imputation: {np.isnan(X).mean():.2%}")
        
        # 5. MAF FILTERING
        print(f"\n[5/7] SNP QC - Minor Allele Frequency (MAF > {self.config.MAF_THRESHOLD})")
        
        allele_freq = np.mean(X, axis=0) / 2.0
        maf = np.minimum(allele_freq, 1 - allele_freq)
        snp_pass_maf = maf > self.config.MAF_THRESHOLD
        
        n_removed_maf = (~snp_pass_maf).sum()
        print(f"   Rimossi {n_removed_maf} SNPs con MAF ≤ {self.config.MAF_THRESHOLD}")
        
        X = X[:, snp_pass_maf]
        bim = bim[snp_pass_maf].reset_index(drop=True)
        
        # 6. HARDY-WEINBERG EQUILIBRIUM
        print(f"\n[6/7] SNP QC - Hardy-Weinberg Equilibrium (P > {self.config.HWE_PVALUE})")
        
        hwe_pass = self._hwe_filter(X, self.config.HWE_PVALUE)
        
        n_removed_hwe = (~hwe_pass).sum()
        print(f"   Rimossi {n_removed_hwe} SNPs che violano HWE (P < {self.config.HWE_PVALUE})")
        
        X = X[:, hwe_pass]
        bim = bim[hwe_pass].reset_index(drop=True)
        
        # 7. LD PRUNING
        if self.config.LD_PRUNING:
            print(f"\n[7/7] LD Pruning (r² > {self.config.LD_THRESHOLD}, window = {self.config.LD_WINDOW_KB}kb)")
            X, bim = self._ld_prune(X, bim)
        else:
            print("\n[7/7] LD Pruning disabilitato")
        
        # FINAL STATS
        print("\n" + "="*80)
        print("📈 STATISTICHE FINALI POST-QC")
        print("="*80)
        print(f"   Samples finali: {X.shape[0]}")
        print(f"   SNPs finali: {X.shape[1]}")
        print(f"   Densità media: 1 SNP ogni ~{2.85e9 / X.shape[1]:.0f} bp")
        print(f"   Compressione: {53347 / X.shape[1]:.1f}x riduzione")
        
        self.num_snps = X.shape[1]
        self.num_samples = X.shape[0]
        self.bim_filtered = bim
        self.fam_filtered = fam
        
        # STANDARDIZATION
        print("\n📊 Standardizzazione dati (mean=0, std=1)...")
        X = self.scaler.fit_transform(X)
        
        # TRAIN/VAL/TEST SPLIT
        print("\n📦 Creazione splits (70% train, 15% val, 15% test)...")
        
        X_train, X_temp = train_test_split(X, test_size=0.3, random_state=42)
        X_val, X_test = train_test_split(X_temp, test_size=0.5, random_state=42)
        
        print(f"   Train: {X_train.shape[0]} samples")
        print(f"   Val:   {X_val.shape[0]} samples")
        print(f"   Test:  {X_test.shape[0]} samples")
        
        self.train_dataset = TensorDataset(torch.FloatTensor(X_train))
        self.val_dataset = TensorDataset(torch.FloatTensor(X_val))
        self.test_dataset = TensorDataset(torch.FloatTensor(X_test))
        
        print("\n✅ Setup completato!\n")
    
    def _hwe_filter(self, X, p_threshold):
        """Hardy-Weinberg Equilibrium test"""
        # Se soglia = 0, salta il test HWE
        if p_threshold == 0:
            print("   ⚠️  HWE test disabilitato per dataset multi-razza")
            return np.ones(X.shape[1], dtype=bool)
        n = X.shape[0]
        hwe_pvalues = np.ones(X.shape[1])
        
        for i in range(X.shape[1]):
            genotypes = X[:, i]
            
            n_AA = np.sum(genotypes == 0)
            n_Aa = np.sum(genotypes == 1)
            n_aa = np.sum(genotypes == 2)
            
            n_total = n_AA + n_Aa + n_aa
            if n_total == 0:
                continue
            
            p = (2*n_AA + n_Aa) / (2*n_total)
            q = 1 - p
            
            exp_AA = n_total * p**2
            exp_Aa = n_total * 2*p*q
            exp_aa = n_total * q**2
            
            if exp_AA > 0 and exp_Aa > 0 and exp_aa > 0:
                chi2 = ((n_AA - exp_AA)**2 / exp_AA + 
                       (n_Aa - exp_Aa)**2 / exp_Aa + 
                       (n_aa - exp_aa)**2 / exp_aa)
                
                hwe_pvalues[i] = 1 - chi2_dist.cdf(chi2, df=1)
        
        return hwe_pvalues > p_threshold
    
    def _ld_prune(self, X, bim):
        """LD pruning basato su distanza fisica"""
        print(f"   Inizio LD pruning su {X.shape[1]} SNPs...")
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"   Device: {device}")
        
        X_std = (X - np.mean(X, axis=0)) / (np.std(X, axis=0) + 1e-8)
        X_tensor = torch.tensor(X_std, device=device, dtype=torch.float32)
        
        n_samples, n_snps = X.shape
        to_keep = np.ones(n_snps, dtype=bool)
        
        window_kb = self.config.LD_WINDOW_KB
        
        for chrom in bim['chrom'].unique():
            chrom_mask = bim['chrom'] == chrom
            chrom_indices = np.where(chrom_mask)[0]
            chrom_pos = bim.loc[chrom_mask, 'pos'].values
            
            i = 0
            while i < len(chrom_indices):
                if not to_keep[chrom_indices[i]]:
                    i += 1
                    continue
                
                pivot_idx = chrom_indices[i]
                pivot_pos = chrom_pos[i]
                
                window_mask = (chrom_pos > pivot_pos) & (chrom_pos <= pivot_pos + window_kb * 1000)
                window_indices = chrom_indices[window_mask]
                window_indices = window_indices[to_keep[window_indices]]
                
                if len(window_indices) == 0:
                    i += 1
                    continue
                
                pivot = X_tensor[:, pivot_idx].unsqueeze(1)
                targets = X_tensor[:, window_indices]
                
                corrs = torch.mm(pivot.t(), targets).squeeze(0) / n_samples
                r2 = corrs ** 2
                
                remove_mask = r2 > self.config.LD_THRESHOLD
                to_keep[window_indices[remove_mask.cpu().numpy()]] = False
                
                i += 1
        
        n_removed = (~to_keep).sum()
        print(f"   ✓ Rimossi {n_removed} SNPs in LD")
        print(f"   ✓ SNPs indipendenti rimanenti: {to_keep.sum()}")
        
        return X[:, to_keep], bim[to_keep].reset_index(drop=True)
    
    def train_dataloader(self):
        return DataLoader(
            self.train_dataset, 
            batch_size=self.config.BATCH_SIZE, 
            shuffle=True, 
            num_workers=self.config.NUM_WORKERS,
            pin_memory=True
        )
    
    def val_dataloader(self):
        return DataLoader(
            self.val_dataset, 
            batch_size=self.config.BATCH_SIZE, 
            num_workers=self.config.NUM_WORKERS,
            pin_memory=True
        )
    
    def test_dataloader(self):
        return DataLoader(
            self.test_dataset, 
            batch_size=self.config.BATCH_SIZE, 
            num_workers=self.config.NUM_WORKERS,
            pin_memory=True
        )

## 5. Modello VAE Standalone

In [14]:
class StandaloneVAE(nn.Module):
    """
    Variational Autoencoder per compressione genomica unsupervised
    
    Architecture:
        Encoder: SNPs → [2048] → [512] → latent_dim (mu, logvar)
        Decoder: latent_dim → [512] → [2048] → SNPs
    """
    
    def __init__(self, num_snps, latent_dim=128, hidden_dims=[2048, 512], dropout=0.1):
        super().__init__()
        self.num_snps = num_snps
        self.latent_dim = latent_dim
        
        # ENCODER
        encoder_layers = []
        in_dim = num_snps
        
        for h_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(in_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.LeakyReLU(0.2),
                nn.Dropout(dropout)
            ])
            in_dim = h_dim
        
        self.encoder = nn.Sequential(*encoder_layers)
        
        self.fc_mu = nn.Linear(hidden_dims[-1], latent_dim)
        self.fc_logvar = nn.Linear(hidden_dims[-1], latent_dim)
        
        # DECODER
        decoder_layers = []
        in_dim = latent_dim
        
        for h_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(in_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.LeakyReLU(0.2),
                nn.Dropout(dropout)
            ])
            in_dim = h_dim
        
        decoder_layers.append(nn.Linear(hidden_dims[0], num_snps))
        self.decoder = nn.Sequential(*decoder_layers)
    
    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar
    
    def get_embeddings(self, x):
        with torch.no_grad():
            mu, _ = self.encode(x)
        return mu

## 6. Lightning Module

In [15]:
class VAELightningModule(pl.LightningModule):
    def __init__(self, config, num_snps):
        super().__init__()
        self.save_hyperparameters()
        self.config = config
        
        self.model = StandaloneVAE(
            num_snps=num_snps,
            latent_dim=config.LATENT_DIM,
            hidden_dims=config.HIDDEN_DIMS,
            dropout=config.DROPOUT_RATE
        )
        
        self.beta = config.BETA
    
    def forward(self, x):
        return self.model(x)
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(), 
            lr=self.config.LEARNING_RATE
        )
        
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=10
        )
        
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss'
            }
        }
    
    def vae_loss(self, x_recon, x, mu, logvar):
        mse_loss = F.mse_loss(x_recon, x, reduction='mean')
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        kl_loss /= x.size(0)
        total_loss = mse_loss + self.beta * kl_loss
        return total_loss, mse_loss, kl_loss
    
    def _common_step(self, batch, batch_idx, stage):
        x = batch[0]
        x_recon, mu, logvar = self(x)
        total_loss, mse_loss, kl_loss = self.vae_loss(x_recon, x, mu, logvar)
        
        self.log(f'{stage}_loss', total_loss, prog_bar=True, on_epoch=True)
        self.log(f'{stage}_mse', mse_loss, prog_bar=True, on_epoch=True)
        self.log(f'{stage}_kl', kl_loss, prog_bar=False, on_epoch=True)
        
        return total_loss
    
    def training_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, 'train')
    
    def validation_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, 'val')
    
    def test_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, 'test')

## 7. Training

In [16]:
def train_vae(config):
    print("\n" + "="*80)
    print("🚀 INIZIO TRAINING VAE")
    print("="*80)
    
    dm = AdaptMapDataModule(config)
    dm.setup()
    
    model = VAELightningModule(config, dm.num_snps)
    
    print(f"\n📐 Architettura Modello:")
    print(f"   Input:  {dm.num_snps} SNPs")
    print(f"   Hidden: {config.HIDDEN_DIMS}")
    print(f"   Latent: {config.LATENT_DIM} dimensions")
    print(f"   Parametri totali: {sum(p.numel() for p in model.parameters()):,}")
    
    checkpoint_callback = ModelCheckpoint(
        dirpath=config.CHECKPOINT_DIR,
        filename='vae-{epoch:02d}-{val_loss:.4f}',
        monitor='val_loss',
        mode='min',
        save_top_k=3,
        save_last=True,
        verbose=True
    )
    
    early_stop_callback = EarlyStopping(
        monitor='val_loss',
        patience=20,
        mode='min',
        verbose=True
    )
    
    logger = TensorBoardLogger(
        save_dir=config.OUTPUT_DIR,
        name='tensorboard_logs'
    )
    
    trainer = pl.Trainer(
        max_epochs=config.MAX_EPOCHS,
        accelerator=config.ACCELERATOR,
        devices=1,
        callbacks=[checkpoint_callback, early_stop_callback],
        logger=logger,
        log_every_n_steps=10,
        gradient_clip_val=1.0,
        deterministic=False
    )
    
    print("\n🏋️ Training in corso...\n")
    trainer.fit(model, dm)
    
    print("\n🧪 Test finale...\n")
    trainer.test(model, dm, ckpt_path='best')
    
    print(f"\n✅ Training completato!")
    print(f"   Best checkpoint: {checkpoint_callback.best_model_path}")
    print(f"   Best val_loss: {checkpoint_callback.best_model_score:.4f}")
    
    return trainer, model, dm, checkpoint_callback.best_model_path

## 8. Funzioni Utility per Embeddings

In [17]:
def extract_embeddings(checkpoint_path, datamodule, output_dir):
    print("\n" + "="*80)
    print("📤 ESTRAZIONE EMBEDDINGS")
    print("="*80)
    
    model = VAELightningModule.load_from_checkpoint(checkpoint_path)
    model.eval()
    model.freeze()
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    results = {}
    
    for split_name, loader in [
        ('train', datamodule.train_dataloader()),
        ('val', datamodule.val_dataloader()),
        ('test', datamodule.test_dataloader())
    ]:
        print(f"\n[{split_name.upper()}] Estrazione in corso...")
        
        embeddings_list = []
        reconstructions_list = []
        
        with torch.no_grad():
            for batch in loader:
                x = batch[0].to(device)
                mu, _ = model.model.encode(x)
                embeddings_list.append(mu.cpu().numpy())
                
                x_recon = model.model.decode(mu)
                reconstructions_list.append(x_recon.cpu().numpy())
        
        embeddings = np.vstack(embeddings_list)
        reconstructions = np.vstack(reconstructions_list)
        original_data = loader.dataset.tensors[0].numpy()
        recon_error = np.mean((original_data - reconstructions)**2)
        
        results[split_name] = {
            'embeddings': embeddings,
            'reconstructions': reconstructions,
            'original': original_data,
            'recon_mse': recon_error
        }
        
        print(f"   ✓ Shape: {embeddings.shape}")
        print(f"   ✓ Reconstruction MSE: {recon_error:.4f}")
        
        np.save(os.path.join(output_dir, f'embeddings_{split_name}.npy'), embeddings)
    
    print(f"\n✅ Embeddings salvati in: {output_dir}/")
    return results


def analyze_embeddings(embeddings_data, output_dir):
    print("\n" + "="*80)
    print("📊 ANALISI EMBEDDINGS")
    print("="*80)
    
    train_emb = embeddings_data['train']['embeddings']
    
    # PCA
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2)
    emb_pca = pca.fit_transform(train_emb)
    
    plt.figure(figsize=(10, 8))
    plt.scatter(emb_pca[:, 0], emb_pca[:, 1], s=10, alpha=0.5)
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    plt.title('PCA degli Embeddings VAE')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'embeddings_pca.png'), dpi=300)
    plt.close()
    print(f"   ✓ PCA plot salvato")
    
    # UMAP
    try:
        import umap
        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
        emb_umap = reducer.fit_transform(train_emb)
        
        plt.figure(figsize=(10, 8))
        plt.scatter(emb_umap[:, 0], emb_umap[:, 1], s=10, alpha=0.5, 
                   c=range(len(emb_umap)), cmap='viridis')
        plt.xlabel('UMAP 1')
        plt.ylabel('UMAP 2')
        plt.title('UMAP degli Embeddings VAE')
        plt.colorbar(label='Sample index')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'embeddings_umap.png'), dpi=300)
        plt.close()
        print(f"   ✓ UMAP plot salvato")
    except ImportError:
        print("   ⚠️  UMAP non disponibile")
    
    print(f"\n   Statistiche Dimensioni Latenti:")
    print(f"   Mean (primi 5): {np.mean(train_emb, axis=0)[:5]}")
    print(f"   Std (primi 5):  {np.std(train_emb, axis=0)[:5]}")
    
    print(f"\n   Qualità Ricostruzione:")
    for split in ['train', 'val', 'test']:
        mse = embeddings_data[split]['recon_mse']
        print(f"   {split.upper()}: MSE = {mse:.4f}")


def downstream_clustering_example(embeddings_data, output_dir):
    print("\n" + "="*80)
    print("🎯 ESEMPIO DOWNSTREAM: CLUSTERING")
    print("="*80)
    
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    from sklearn.decomposition import PCA
    
    train_emb = embeddings_data['train']['embeddings']
    n_clusters = 20
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(train_emb)
    silhouette = silhouette_score(train_emb, clusters)
    
    print(f"\n   K-means con {n_clusters} clusters:")
    print(f"   Silhouette Score: {silhouette:.3f}")
    
    pca = PCA(n_components=2)
    emb_pca = pca.fit_transform(train_emb)
    
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(emb_pca[:, 0], emb_pca[:, 1], c=clusters, 
                         cmap='tab20', s=10, alpha=0.6)
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    plt.title(f'K-means Clustering (k={n_clusters})')
    plt.colorbar(scatter, label='Cluster')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'clustering_example.png'), dpi=300)
    plt.close()
    print(f"   ✓ Clustering plot salvato")

## 9. ESECUZIONE COMPLETA

In [ ]:
# STEP 1: Training
trainer, model, datamodule, best_ckpt = train_vae(config)


🚀 INIZIO TRAINING VAE

📊 CARICAMENTO E QUALITY CONTROL DATASET ADAPTMAP

[1/7] Caricamento file PLINK: ./data/ADAPTmap_genotypeTOP_20160222_full.bed
   Shape iniziale: (4653, 53347) (4653 samples, 53347 SNPs)
   Missing rate globale: 3.03%

[2/7] Sample Quality Control (MIND < 0.05)
   Shape iniziale: (4653, 53347) (4653 samples, 53347 SNPs)
   Missing rate globale: 3.03%

[2/7] Sample Quality Control (MIND < 0.05)
   Rimossi 231 samples con call rate < 95.00%
   ✓ Samples rimanenti: 4422

[3/7] SNP Quality Control - Missingness (GENO < 0.05)
   Rimossi 231 samples con call rate < 95.00%
   ✓ Samples rimanenti: 4422

[3/7] SNP Quality Control - Missingness (GENO < 0.05)
   Rimossi 2595 SNPs con call rate < 95.00%
   Rimossi 2595 SNPs con call rate < 95.00%

[4/7] Imputation valori mancanti (strategia: mean)

[4/7] Imputation valori mancanti (strategia: mean)
   ✓ Missing rate post-imputation: 0.00%

[5/7] SNP QC - Minor Allele Frequency (MAF > 0.01)
   Rimossi 4 SNPs con MAF ≤ 0.01

[

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



📐 Architettura Modello:
   Input:  49923 SNPs
   Hidden: [2048, 512]
   Latent: 128 dimensions
   Parametri totali: 206,843,907

🏋️ Training in corso...




  | Name  | Type          | Params | Mode 
------------------------------------------------
0 | model | StandaloneVAE | 206 M  | train
------------------------------------------------
206 M     Trainable params
0         Non-trainable params
206 M     Total params
827.376   Total estimated model params size (MB)
22        Modules in train mode
0         Modules in eval mode


Epoch 0: 100%|██████████| 49/49 [00:01<00:00, 30.29it/s, v_num=5, train_loss_step=0.949, train_mse_step=0.939, val_loss=0.912, val_mse=0.900, train_loss_epoch=1.020, train_mse_epoch=1.010]

...

Metric val_loss improved by 0.001 >= min_delta = 0.0. New best score: 0.791
Epoch 32, global step 1617: 'val_loss' reached 0.79071 (best 0.79071), saving model to 'C:\\Users\\losp0\\Documents\\Uni\\goat-latent-space-project\\output_vae_standalone\\checkpoints\\vae-epoch=32-val_loss=0.7907.ckpt' as top 3
Epoch 32, global step 1617: 'val_loss' reached 0.79071 (best 0.79071), saving model to 'C:\\Users\\losp0\\Documents\\Uni\\goat-latent-space-project\\output_vae_standalone\\checkpoints\\vae-epoch=32-val_loss=0.7907.ckpt' as top 3


Epoch 33: 100%|██████████| 49/49 [00:01<00:00, 27.96it/s, v_num=5, train_loss_step=0.806, train_mse_step=0.795, val_loss=0.790, val_mse=0.779, train_loss_epoch=0.792, train_mse_epoch=0.781]

Metric val_loss improved by 0.001 >= min_delta = 0.0. New best score: 0.790
Epoch 33, global step 1666: 'val_loss' reached 0.78976 (best 0.78976), saving model to 'C:\\Users\\losp0\\Documents\\Uni\\goat-latent-space-project\\output_vae_standalone\\checkpoints\\vae-epoch=33-val_loss=0.7898.ckpt' as top 3
Epoch 33, global step 1666: 'val_loss' reached 0.78976 (best 0.78976), saving model to 'C:\\Users\\losp0\\Documents\\Uni\\goat-latent-space-project\\output_vae_standalone\\checkpoints\\vae-epoch=33-val_loss=0.7898.ckpt' as top 3


Epoch 34: 100%|██████████| 49/49 [00:01<00:00, 26.49it/s, v_num=5, train_loss_step=0.774, train_mse_step=0.763, val_loss=0.789, val_mse=0.779, train_loss_epoch=0.791, train_mse_epoch=0.780]

Metric val_loss improved by 0.000 >= min_delta = 0.0. New best score: 0.789
Epoch 34, global step 1715: 'val_loss' reached 0.78932 (best 0.78932), saving model to 'C:\\Users\\losp0\\Documents\\Uni\\goat-latent-space-project\\output_vae_standalone\\checkpoints\\vae-epoch=34-val_loss=0.7893.ckpt' as top 3
Epoch 34, global step 1715: 'val_loss' reached 0.78932 (best 0.78932), saving model to 'C:\\Users\\losp0\\Documents\\Uni\\goat-latent-space-project\\output_vae_standalone\\checkpoints\\vae-epoch=34-val_loss=0.7893.ckpt' as top 3


In [ ]:
# STEP 2: Estrai Embeddings
embeddings_data = extract_embeddings(
    checkpoint_path=best_ckpt,
    datamodule=datamodule,
    output_dir=config.EMBEDDINGS_DIR
)


📤 ESTRAZIONE EMBEDDINGS

[TRAIN] Estrazione in corso...
   ✓ Shape: (3095, 128)
   ✓ Reconstruction MSE: 1.3931

[VAL] Estrazione in corso...
   ✓ Shape: (663, 128)
   ✓ Reconstruction MSE: 0.7158

[TEST] Estrazione in corso...
   ✓ Shape: (664, 128)
   ✓ Reconstruction MSE: 0.7137

✅ Embeddings salvati in: ./output_vae_standalone\embeddings/

[TRAIN] Estrazione in corso...
   ✓ Shape: (3095, 128)
   ✓ Reconstruction MSE: 1.3931

[VAL] Estrazione in corso...
   ✓ Shape: (663, 128)
   ✓ Reconstruction MSE: 0.7158

[TEST] Estrazione in corso...
   ✓ Shape: (664, 128)
   ✓ Reconstruction MSE: 0.7137

✅ Embeddings salvati in: ./output_vae_standalone\embeddings/


In [ ]:
# STEP 3: Analizza Embeddings
analyze_embeddings(embeddings_data, config.OUTPUT_DIR)


📊 ANALISI EMBEDDINGS
   ✓ PCA plot salvato
   ✓ PCA plot salvato
   ✓ UMAP plot salvato

   Statistiche Dimensioni Latenti:
   Mean (primi 5): [-0.04569495 -0.0019207   0.0023515  -0.03097182 -0.02198097]
   Std (primi 5):  [0.92376775 0.9081391  0.9360837  0.93003345 0.9573902 ]

   Qualità Ricostruzione:
   TRAIN: MSE = 1.3931
   VAL: MSE = 0.7158
   TEST: MSE = 0.7137
   ✓ UMAP plot salvato

   Statistiche Dimensioni Latenti:
   Mean (primi 5): [-0.04569495 -0.0019207   0.0023515  -0.03097182 -0.02198097]
   Std (primi 5):  [0.92376775 0.9081391  0.9360837  0.93003345 0.9573902 ]

   Qualità Ricostruzione:
   TRAIN: MSE = 1.3931
   VAL: MSE = 0.7158
   TEST: MSE = 0.7137


In [ ]:
# STEP 4: Esempio Downstream
downstream_clustering_example(embeddings_data, config.OUTPUT_DIR)


🎯 ESEMPIO DOWNSTREAM: CLUSTERING

   K-means con 20 clusters:
   Silhouette Score: 0.030

   K-means con 20 clusters:
   Silhouette Score: 0.030
   ✓ Clustering plot salvato
   ✓ Clustering plot salvato


## 10. Utilizzo Embeddings per Nuovi Task

Gli embeddings salvati possono essere caricati e usati per:
- Classificazione razze
- Predizione fenotipi
- GWAS compresso
- Transfer learning su nuovi dataset

In [ ]:
# Esempio: Carica embeddings salvati
train_embeddings = np.load(os.path.join(config.EMBEDDINGS_DIR, 'embeddings_train.npy'))
test_embeddings = np.load(os.path.join(config.EMBEDDINGS_DIR, 'embeddings_test.npy'))

print(f"Train embeddings shape: {train_embeddings.shape}")
print(f"Test embeddings shape: {test_embeddings.shape}")

# Usa per classificazione con Random Forest
# from sklearn.ensemble import RandomForestClassifier
# clf = RandomForestClassifier()
# clf.fit(train_embeddings, y_train)
# predictions = clf.predict(test_embeddings)

Train embeddings shape: (3095, 128)
Test embeddings shape: (664, 128)


## Fine Notebook

✅ **Output generati:**
- Checkpoints VAE in `./output_vae_standalone/checkpoints/`
- Embeddings in `./output_vae_standalone/embeddings/`
- Visualizzazioni in `./output_vae_standalone/*.png`
- TensorBoard logs in `./output_vae_standalone/tensorboard_logs/`

📊 **Per visualizzare i logs di training:**
```bash
tensorboard --logdir=./output_vae_standalone/tensorboard_logs/
```